In [1]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# ШАГ 1. Загрузка исходных данных
# ==============================================================================
input_path = r"C:\Users\User\rstudio-docker\Pharma_clean_1.csv"
output_path = r"C:\Users\User\rstudio-docker\Pharma_clean_2.csv"

print("⏳ Загрузка данных...")
df = pd.read_csv(input_path)

# ==============================================================================
# ШАГ 2. Стандартизация названий столбцов (snake_case)
# ==============================================================================
# Приводим к нижнему регистру и заменяем пробелы, слэши, точки и дефисы на нижние подчеркивания
df.columns = (df.columns.str.strip()
              .str.lower()
              .str.replace(' ', '_')
              .str.replace('/', '_')
              .str.replace('.', '_')
              .str.replace('-', '_'))

# ==============================================================================
# ШАГ 3. Обработка дат и сортировка
# ==============================================================================
# Преобразуем столбец 'дата' из строки в формат datetime (формат в файле: Месяц/День/Год)
df['дата'] = pd.to_datetime(df['дата'], format='%m/%d/%Y')

# Сортируем данные по дате по возрастанию для корректной хронологии
df = df.sort_values('дата').reset_index(drop=True)

# ==============================================================================
# ШАГ 4. Feature Engineering (Создание признаков для BI-дашборда)
# ==============================================================================
print("⏳ Создание новых признаков для дашборда...")

# Базовые признаки из даты
df['год'] = df['дата'].dt.year
df['месяц'] = df['дата'].dt.month
df['квартал'] = df['дата'].dt.quarter
df['день_месяца'] = df['дата'].dt.day
df['день_недели'] = df['дата'].dt.day_name() # Вернет 'Monday', 'Tuesday' и т.д.

# Признак "Сезон"
def get_season(month):
    if month in [12, 1, 2]: return 'Зима'
    if month in [3, 4, 5]: return 'Весна'
    if month in [6, 7, 8]: return 'Лето'
    return 'Осень'

df['сезон'] = df['месяц'].apply(get_season)

# Признак "Выходной день" (1 - да, 0 - нет)
df['is_weekend'] = df['день_недели'].isin(['Saturday', 'Sunday']).astype(int)

# Признак "Нулевые продажи" (помогает в DataLens отфильтровать дни, когда аптека не работала)
drug_cols = [
    'нпвс_пр__уксус', 'нпвс_пр__проп', 'салиц__к_та', 'парацетамол',
    'психолепт_успокоит', 'снотворн_седавтивн', 'противоастмат_противокашл', 'антигистамин'
]
df['is_all_zeros'] = (df[drug_cols] == 0).all(axis=1).astype(int)

# ==============================================================================
# ШАГ 5. Сохранение финального файла
# ==============================================================================
print(f"⏳ Сохранение файла в: {output_path}")
df.to_csv(output_path, index=False, encoding='utf-8-sig') 
# encoding='utf-8-sig' гарантирует, что русские буквы корректно откроются в Excel

print("✅ ГОТОВО! Файл 'Pharma_clean_2.csv' успешно создан.")
print("\n📊 Проверка первых 3 строк нового файла:")
print(df.head(3))
print("\n📋 Итоговый список столбцов:")
print(df.columns.tolist())

⏳ Загрузка данных...
⏳ Создание новых признаков для дашборда...
⏳ Сохранение файла в: C:\Users\User\rstudio-docker\Pharma_clean_2.csv
✅ ГОТОВО! Файл 'Pharma_clean_2.csv' успешно создан.

📊 Проверка первых 3 строк нового файла:
        дата  нпвс_пр__уксус  нпвс_пр__проп  салиц__к_та  парацетамол  \
0 2014-01-02               0              4            3           32   
1 2014-01-03               8              4            4           51   
2 2014-01-04               2              1            6           62   

   психолепт_успокоит  снотворн_седавтивн  противоастмат_противокашл  \
0                   7                   0                          0   
1                  16                   0                         20   
2                  10                   0                          9   

   антигистамин   год  месяц день_недели  квартал  день_месяца сезон  \
0             2  2014      1    Thursday        1            2  Зима   
1             4  2014      1      Friday       